In [ ]:
# ==========================================
# INSTALL DEPENDENCIES
# ==========================================
!pip install -q firebase-admin transformers torch lime

# ==========================================
# IMPORTS
# ==========================================
import os
import torch
import torch.nn.functional as F
import firebase_admin
from firebase_admin import credentials, firestore
from transformers import BertTokenizer, BertForSequenceClassification
from lime.lime_text import LimeTextExplainer

print("Libraries loaded")

# ==========================================
# FIREBASE INIT
# ==========================================
cred = credentials.Certificate("/kaggle/input/datasets/prophisher/firebase-key/firebase_key.json")
firebase_admin.initialize_app(cred)
db = firestore.client()
print("Connected to Firestore")
docs = list(db.collection("emails").stream())
print("Total emails in Firestore:", len(docs))

# ==========================================
# LOAD BERT MODEL
# ==========================================
MODEL_PATH = "/kaggle/input/datasets/prophisher/bert-model"

tokenizer = BertTokenizer.from_pretrained(MODEL_PATH)
model = BertForSequenceClassification.from_pretrained(MODEL_PATH)
model.eval()
print("BERT model loaded")

# ==========================================
# LIME EXPLAINER
# ==========================================
explainer = LimeTextExplainer(class_names=["Legitimate", "Phishing"])

# ==========================================
# PREDICTION FUNCTION FOR LIME
# ==========================================
def predict_proba(texts):
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    return probs.cpu().numpy()

# ==========================================
# GENERATE LIME EXPLANATION
# ==========================================
def generate_lime(text):
    explanation = explainer.explain_instance(
        text,
        predict_proba,
        num_features=10,
        num_samples=100
    )
    results = []
    for word, score in explanation.as_list():
        results.append({
            "word": word,
            "score": float(score)
        })
    return results

# ==========================================
# FETCH EMAILS FROM FIRESTORE
# Only process emails without explanation
# ==========================================

BATCH_SIZE = 20
emails = list(db.collection("emails").stream())

print("Fetching emails...")

processed = 0

for doc in emails:

    if processed >= BATCH_SIZE:
        break

    data = doc.to_dict()

    # Skip only if explanation already exists
    if data.get("lime_explanation") not in [None, []]:
        continue

    subject = data.get("subject", "")
    content = data.get("content", "")

    text = (subject + " " + content).strip()

    if len(text) < 5:
        print("Skipping empty email:", doc.id)
        continue

    try:
        explanation = generate_lime(text)

        db.collection("emails").document(doc.id).update({
            "lime_explanation": explanation
        })

        print("Updated:", doc.id)
        processed += 1

    except Exception as e:
        print("Error processing:", doc.id, e)

print("Batch finished. Emails updated:", processed)